# Parametric copula families

This notebook reproduces every result quoted in the text on the parametric copula families with `pyvinecopulib`, the library used in the thesis:

1. the table of families (symmetry, asymptotic tail dependence),
2. the finite-threshold joint-extreme probabilities that the paragraph below the table quotes (e.g. 0.40 versus 0.22),
3. the tail orders behind the comparisons between Gaussian and Frank, and between Gumbel and Joe,
4. a check that each number in the text matches the computed value, and sensitivity of the numbers to Kendall's $\tau$ and to the Student-$t$ degrees of freedom.

All families are compared at the same Kendall's $\tau$, so that differences are differences in *shape*, not in overall dependence strength. No data are used; everything follows from the copula definitions.

In [1]:
import numpy as np
import pandas as pd
import pyvinecopulib as pv
from scipy.stats import t as _t

F = pv.BicopFamily

# ---- settings ----------------------------------------
TAU = 0.5      # common Kendall's tau at which the families are compared
NU = 4         # Student-t degrees of freedom (an assumption, see sensitivity below)
Q = 0.95       # threshold for the joint-extreme probabilities in the text
# ----------------------------------------------------------------------------

def taildep(b):
    """Asymptotic tail dependence coefficients (lambda_L, lambda_U) of a bivariate copula, from closed forms.

    Closed forms keep the result independent of the pyvinecopulib version. Gaussian and Frank have no tail dependence, the Student-t
    copula has lambda = 2 t_{nu+1}(-sqrt((nu + 1)(1 - rho)/(1 + rho))) in both tails (Demarta & McNeil, 2005, eq. 13), Clayton has
    lambda_L = 2**(-1/theta), Gumbel and Joe have lambda_U = 2 - 2**(1/theta) (Joe, 2014; Nelsen, 2006). A rotation of 180 degrees
    swaps the two tails.

    Demarta, S., & McNeil, A. J. (2005). The t copula and related copulas. International Statistical Review, 73(1), 111-129.
    Joe, H. (2014). Dependence modeling with copulas. CRC Press.
    Nelsen, R. B. (2006). An introduction to copulas (2nd ed.). Springer.

    Parameters:
    b (pv.Bicop): Fitted or constructed bivariate copula of one of the families Gaussian, Student, Clayton, Gumbel, Joe, Frank.

    Returns:
    tuple: (lambda_L, lambda_U) as floats.

    Raises:
    ValueError: If the family of b is not one of the families above.
    """
    fam, rot = b.family, b.rotation
    th = float(np.ravel(b.parameters)[0])
    if fam == F.gaussian or fam == F.frank:
        lam = (0.0, 0.0)
    elif fam == F.student:
        rho, nu = float(b.parameters[0, 0]), float(b.parameters[1, 0])
        l = 2 * _t.cdf(-np.sqrt((nu + 1) * (1 - rho) / (1 + rho)), nu + 1)
        lam = (l, l)
    elif fam == F.clayton:
        lam = (2 ** (-1 / th), 0.0)
    elif fam in (F.gumbel, F.joe):
        lam = (0.0, 2 - 2 ** (1 / th))
    else:
        raise ValueError(fam)
    return (lam[1], lam[0]) if rot == 180 else lam

def kendall_tau(b):
    """Kendall's tau of a bivariate copula, across pyvinecopulib versions.

    Parameters:
    b (pv.Bicop): Bivariate copula.

    Returns:
    float: Kendall's tau of b.
    """
    try:
        return b.tau
    except AttributeError:
        return float(b.parameters_to_tau(b.parameters))

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("pyvinecopulib", pv.__version__)

pyvinecopulib 0.7.5


## 1. The seven families at a common Kendall's $\tau$

Each family's parameter is set so that its Kendall's $\tau$ equals `TAU`. For the Student-$t$ copula the correlation is $\rho=\sin(\pi\tau/2)$ and the degrees of freedom are fixed at `NU`. The rotated Gumbel is the Gumbel copula with `rotation=180`, which swaps the lower and the upper tail.

In [2]:
SPEC = [
    ("Gaussian",      F.gaussian, 0),
    ("Student-t",     F.student,  0),
    ("Clayton",       F.clayton,  0),
    ("Gumbel",        F.gumbel,   0),
    ("Gumbel (180°)", F.gumbel,   180),
    ("Frank",         F.frank,    0),
    ("Joe",           F.joe,      0),
]

def build(fam, rotation=0, tau=TAU, nu=NU):
    """Bivariate copula of one family with a given Kendall's tau.

    For the Student-t copula the correlation is rho = sin(pi tau / 2) (inverse of Demarta & McNeil, 2005, eq. 9) and the degrees of freedom are nu;
    for the other families the parameter is obtained from tau by the library.

    Demarta, S., & McNeil, A. J. (2005). The t copula and related copulas. International Statistical Review, 73(1), 111-129.

    Parameters:
    fam (pv.BicopFamily): Copula family.
    rotation (int): Rotation in degrees, 0 or 180. Default 0.
    tau (float): Kendall's tau. Default TAU.
    nu (float): Degrees of freedom of the Student-t copula, ignored for the other families. Default NU.

    Returns:
    pv.Bicop: Copula with Kendall's tau equal to tau.
    """
    if fam == F.student:
        rho = np.sin(np.pi * tau / 2)
        return pv.Bicop(F.student, rotation, np.array([[rho], [nu]]))
    base = pv.Bicop(fam, rotation)
    return pv.Bicop(fam, rotation, base.tau_to_parameters(tau))

def families(tau=TAU, nu=NU):
    """The seven copulas of SPEC at a common Kendall's tau.

    Parameters:
    tau (float): Kendall's tau. Default TAU.
    nu (float): Degrees of freedom of the Student-t copula. Default NU.

    Returns:
    dict: Family name -> pv.Bicop, in the order of SPEC.
    """
    return {name: build(fam, rot, tau, nu) for name, fam, rot in SPEC}

FAMS = families()

pd.DataFrame(
    {
        "parameter(s)": {n: ", ".join(f"{p:.3f}" for p in b.parameters.ravel()) for n, b in FAMS.items()},
        "Kendall's tau": {n: kendall_tau(b) for n, b in FAMS.items()},
    }
)

,parameter(s),Kendall's tau
Gaussian,0.707,0.500
Student-t,"0.707, 4.000",0.500
Clayton,2.000,0.500
Gumbel,2.000,0.500
Gumbel (180°),2.000,0.500
Frank,5.736,0.500
Joe,2.856,0.500


## 2. Helper functions

For a copula $C$ with uniform margins,

$$\hat\chi(q)=P(U_2>q\mid U_1>q)=\frac{1-2q+C(q,q)}{1-q},\qquad
\hat\chi_L(q)=P(U_2<1-q\mid U_1<1-q)=\frac{C(1-q,1-q)}{1-q}.$$

Under independence both equal $1-q$. These are the finite-threshold quantities; the thesis reports them as curves on $q\in\{0.80,\dots,0.95\}$.

In [3]:
def cdf(b, u, v):
    """Copula cdf C(u, v) at pairs of points.

    Parameters:
    b (pv.Bicop): Bivariate copula.
    u (float or array-like): First coordinates in (0, 1).
    v (float or array-like): Second coordinates in (0, 1), same length as u.

    Returns:
    np.ndarray: C(u_i, v_i) for every pair.
    """
    u, v = np.atleast_1d(u).astype(float), np.atleast_1d(v).astype(float)
    return b.cdf(np.column_stack([u, v]))

def chi_upper(b, q):
    """Joint upper-extreme probability P(U2 > q | U1 > q) = (1 - 2q + C(q, q)) / (1 - q).

    Parameters:
    b (pv.Bicop): Bivariate copula.
    q (float or array-like): Threshold(s) in (0, 1).

    Returns:
    np.ndarray: Probability at every q.
    """
    q = np.atleast_1d(q).astype(float)
    return (1 - 2 * q + cdf(b, q, q)) / (1 - q)

def chi_lower(b, q):
    """Joint lower-extreme probability P(U2 < 1 - q | U1 < 1 - q) = C(1 - q, 1 - q) / (1 - q).

    Parameters:
    b (pv.Bicop): Bivariate copula.
    q (float or array-like): Threshold(s) in (0, 1).

    Returns:
    np.ndarray: Probability at every q.
    """
    q = np.atleast_1d(q).astype(float)
    return cdf(b, 1 - q, 1 - q) / (1 - q)

def is_symmetric(b, tol=1e-8):
    """Numerical check of radial symmetry, C(u, v) = u + v - 1 + C(1 - u, 1 - v), on a 19 x 19 grid over [0.05, 0.95]^2.

    Parameters:
    b (pv.Bicop): Bivariate copula.
    tol (float): Largest tolerated absolute deviation. Default 1e-8.

    Returns:
    bool: True if the maximum deviation is below tol.
    """
    g = np.linspace(0.05, 0.95, 19)
    U, V = np.meshgrid(g, g)
    u, v = U.ravel(), V.ravel()
    return bool(np.max(np.abs(cdf(b, u, v) - (u + v - 1 + cdf(b, 1 - u, 1 - v)))) < tol)

## 3. The table of families

* **Symmetric**: checked numerically via radial symmetry.
* **Lower / Upper tail dep.**: the asymptotic coefficients $\lambda_L,\lambda_U$ from their closed forms (so the notebook does not depend on the library version); a checkmark means $\lambda>0$ (asymptotic dependence, $\chi>0$).

In [4]:
rows = []
for name, b in FAMS.items():
    lam_L, lam_U = taildep(b)
    rows.append({
        "Family": name,
        "Symmetric": "yes" if is_symmetric(b) else "no",
        "lambda_L": lam_L,
        "lambda_U": lam_U,
        "Lower tail dep.": "yes" if lam_L > 1e-8 else "no",
        "Upper tail dep.": "yes" if lam_U > 1e-8 else "no",
    })
table = pd.DataFrame(rows).set_index("Family")
table

,Symmetric,lambda_L,lambda_U,Lower tail dep.,Upper tail dep.
Family,,,,,
Gaussian,yes,0.000,0.000,no,no
Student-t,no,0.397,0.397,yes,yes
Clayton,no,0.707,0.000,yes,no
Gumbel,no,0.000,0.586,no,yes
Gumbel (180°),no,0.586,0.000,yes,no
Frank,yes,0.000,0.000,no,no
Joe,no,0.000,0.725,no,yes


Every combination of lower and upper asymptotic tail dependence appears at least once: neither tail (Gaussian, Frank), both (Student-$t$), lower only (Clayton, rotated Gumbel) and upper only (Gumbel, Joe).

In [5]:
combos = table.groupby(["Lower tail dep.", "Upper tail dep."]).apply(lambda d: ", ".join(d.index), include_groups=False)
combos

Lower tail dep.  Upper tail dep.
no               no                        Gaussian, Frank
                 yes                           Gumbel, Joe
yes              no                 Clayton, Gumbel (180°)
                 yes                             Student-t
dtype: object

## 4. Joint extremes at a finite threshold

The last column of the table in the thesis describes this in words. Here are the underlying numbers at $q=0.95$ and the given $\tau$, next to the independence benchmark $1-q$.

In [6]:
fin = pd.DataFrame(
    {
        "lower  P(U2<1-q | U1<1-q)": {n: chi_lower(b, Q)[0] for n, b in FAMS.items()},
        "upper  P(U2>q | U1>q)":     {n: chi_upper(b, Q)[0] for n, b in FAMS.items()},
    }
)
fin.loc["(independence)"] = [1 - Q, 1 - Q]
fin

,lower P(U2<1-q | U1<1-q),upper P(U2>q | U1>q)
Gaussian,0.398,0.398
Student-t,0.482,0.482
Clayton,0.708,0.136
Gumbel,0.289,0.601
Gumbel (180°),0.601,0.289
Frank,0.225,0.225
Joe,0.131,0.725
(independence),0.050,0.050


## 5. Tail order

The tail order $\kappa$ is defined by $C(u,u)\sim u^{\kappa}$ in the lower tail (and analogously for the survival function in the upper tail). It separates three cases:

| $\kappa$ | grade | meaning |
|---|---|---|
| $1$ | strong | asymptotic dependence, $\chi>0$ |
| $1<\kappa<2$ | mild | asymptotically independent, but joint extremes fade more slowly than under independence |
| $2$ | none | joint extremes fade as under independence |

The numerical slope is estimated between $u=10^{-3}$ and $u=10^{-5}$ from the library's own CDF, so it is only approximate; for the Gaussian copula the table shows the finite-$u$ slope next to the limit $2/(1+\rho)$. The "limit" column gives the value derived analytically for each family at this $\tau$.

In [7]:
def kappa_numeric(b, u1=1e-3, u2=1e-5):
    """Numerical tail orders of a copula: log-log slope of C(u, u) (lower) and of the joint survival function (upper) between u1 and u2.

    The tail order kappa is defined by C(u, u) ~ u**kappa l(u) with l slowly varying (Hua & Joe, 2011, Definition 2). It is the
    inverse of the coefficient of tail dependence eta of Ledford & Tawn (1996), and for the bivariate Gaussian copula with rho > 0 it
    equals 2/(1 + rho) (Hua & Joe, 2011, Section 2.1; eta = (1 + rho)/2 in Ledford & Tawn, 1996).

    Hua, L., & Joe, H. (2011). Tail order and intermediate tail dependence of multivariate copulas. Journal of Multivariate
    Analysis, 102(10), 1454-1471.
    Ledford, A. W., & Tawn, J. A. (1996). Statistics for near independence in multivariate extreme values. Biometrika, 83(1), 169-187.

    Parameters:
    b (pv.Bicop): Bivariate copula.
    u1 (float): Upper evaluation point of the slope. Default 1e-3.
    u2 (float): Lower evaluation point of the slope. Default 1e-5.

    Returns:
    tuple: (kappa_L, kappa_U) as floats.
    """
    c_low = lambda u: cdf(b, u, u)[0]
    surv  = lambda u: 1 - 2 * (1 - u) + cdf(b, 1 - u, 1 - u)[0]
    kl = (np.log(c_low(u1)) - np.log(c_low(u2))) / (np.log(u1) - np.log(u2))
    ku = (np.log(surv(u1)) - np.log(surv(u2))) / (np.log(u1) - np.log(u2))
    return kl, ku

rho = np.sin(np.pi * TAU / 2)
theta_gumbel = FAMS["Gumbel"].parameters[0, 0]
kappa_limit = {   # derived limits at this tau: (lower, upper)
    "Gaussian":      (2 / (1 + rho), 2 / (1 + rho)),
    "Student-t":     (1.0, 1.0),
    "Clayton":       (1.0, 2.0),
    "Gumbel":        (2 ** (1 / theta_gumbel), 1.0),
    "Gumbel (180°)": (1.0, 2 ** (1 / theta_gumbel)),
    "Frank":         (2.0, 2.0),
    "Joe":           (2.0, 1.0),
}
grade = lambda k: "strong" if abs(k - 1) < 1e-9 else ("none" if abs(k - 2) < 1e-9 else "mild")

rows = []
for name, b in FAMS.items():
    kl, ku = kappa_numeric(b)
    (Ll, Lu) = kappa_limit[name]
    rows.append({"Family": name,
                 "kappa_L (numeric)": kl, "kappa_L (limit)": Ll, "lower grade": grade(Ll),
                 "kappa_U (numeric)": ku, "kappa_U (limit)": Lu, "upper grade": grade(Lu)})
pd.DataFrame(rows).set_index("Family")

,kappa_L (numeric),kappa_L (limit),lower grade,kappa_U (numeric),kappa_U (limit),upper grade
Family,,,,,,
Gaussian,1.205,1.172,mild,1.205,1.172,mild
Student-t,1.005,1.000,strong,1.005,1.000,strong
Clayton,1.000,1.000,strong,2.000,2.000,none
Gumbel,1.414,1.414,mild,1.000,1.000,strong
Gumbel (180°),1.000,1.000,strong,1.414,1.414,mild
Frank,1.999,2.000,none,1.999,2.000,none
Joe,2.000,2.000,none,1.000,1.000,strong


The Gaussian is *mild* in both tails and Frank is *none*: both are asymptotically independent, but the Gaussian's joint extremes fade much more slowly. Gumbel and Joe are *strong* in the upper tail; in the lower tail Gumbel is *mild* and Joe is *none*. This is the structure the text describes.

## 6. Check against the numbers quoted in the text

Each number in the paragraph under the table is recomputed and compared after rounding to two decimals.

In [8]:
quoted = [
    # (description, family, side, value quoted in the text)
    ("Gaussian, upper P(U2>q | U1>q) at q=0.95",      "Gaussian", "upper", 0.40),
    ("Frank,    upper P(U2>q | U1>q) at q=0.95",      "Frank",    "upper", 0.22),
    ("Joe,      upper P(U2>q | U1>q) at q=0.95",      "Joe",      "upper", 0.73),
    ("Gumbel,   upper P(U2>q | U1>q) at q=0.95",      "Gumbel",   "upper", 0.60),
    ("Joe,      lower P(U2<0.05 | U1<0.05)",          "Joe",      "lower", 0.13),
    ("Gumbel,   lower P(U2<0.05 | U1<0.05)",          "Gumbel",   "lower", 0.29),
]
rows = []
for desc, fam, side, val in quoted:
    comp = (chi_upper if side == "upper" else chi_lower)(FAMS[fam], Q)[0]
    rows.append({"quantity": desc, "computed": comp, "quoted": val, "match (2 dp)": round(comp, 2) == val})
rows.append({"quantity": "independence benchmark 1 - q", "computed": 1 - Q, "quoted": 0.05,
             "match (2 dp)": round(1 - Q, 2) == 0.05})
check = pd.DataFrame(rows).set_index("quantity")
display(check)
assert check["match (2 dp)"].all(), "a quoted number in the text does not match the computation"
print("All quoted numbers match.")

# The rotated Gumbel is the mirror image of the Gumbel copula: its lower-tail curve is the Gumbel upper-tail curve
# and vice versa, and its tail coefficients swap.
qq = np.linspace(0.5, 0.99, 50)
assert np.allclose(chi_lower(FAMS["Gumbel (180°)"], qq), chi_upper(FAMS["Gumbel"], qq), atol=1e-9)
assert np.allclose(chi_upper(FAMS["Gumbel (180°)"], qq), chi_lower(FAMS["Gumbel"], qq), atol=1e-9)
assert taildep(FAMS["Gumbel (180°)"]) == taildep(FAMS["Gumbel"])[::-1]
print("Gumbel (180°) is the mirror image of Gumbel: curves and tail coefficients swap.")

,computed,quoted,match (2 dp)
quantity,,,
"Gaussian, upper P(U2>q | U1>q) at q=0.95",0.398,0.400,True
"Frank, upper P(U2>q | U1>q) at q=0.95",0.225,0.220,True
"Joe, upper P(U2>q | U1>q) at q=0.95",0.725,0.730,True
"Gumbel, upper P(U2>q | U1>q) at q=0.95",0.601,0.600,True
"Joe, lower P(U2<0.05 | U1<0.05)",0.131,0.130,True
"Gumbel, lower P(U2<0.05 | U1<0.05)",0.289,0.290,True
independence benchmark 1 - q,0.050,0.050,True


All quoted numbers match.
Gumbel (180°) is the mirror image of Gumbel: curves and tail coefficients swap.


## 7. Sensitivity

**To Kendall's $\tau$.** The ranking of the families (Gaussian above Frank, Joe above Gumbel in the upper tail, and the reverse in the lower tail) should not depend on the particular value $\tau=0.5$. The table shows $\hat\chi(0.95)$ and $\hat\chi_L(0.95)$ for three values of $\tau$.

In [9]:
parts = {}
for tau in (0.2, 0.5, 0.8):
    fams = families(tau=tau)
    parts[f"tau={tau}"] = pd.DataFrame({
        "lower": {n: chi_lower(b, Q)[0] for n, b in fams.items()},
        "upper": {n: chi_upper(b, Q)[0] for n, b in fams.items()},
    })
sens_tau = pd.concat(parts, axis=1)
display(sens_tau.style.format("{:.2f}"))

# the orderings quoted in the text, at every tau
for tau in (0.2, 0.5, 0.8):
    s = sens_tau[f"tau={tau}"]
    ok = (s.loc["Gaussian", "upper"] > s.loc["Frank", "upper"]
          and s.loc["Joe", "upper"] > s.loc["Gumbel", "upper"]
          and s.loc["Gumbel", "lower"] > s.loc["Joe", "lower"])
    print(f"tau={tau}: Gaussian>Frank (upper), Joe>Gumbel (upper), Gumbel>Joe (lower): {ok}")

tau=0.2: Gaussian>Frank (upper), Joe>Gumbel (upper), Gumbel>Joe (lower): True
tau=0.5: Gaussian>Frank (upper), Joe>Gumbel (upper), Gumbel>Joe (lower): True
tau=0.8: Gaussian>Frank (upper), Joe>Gumbel (upper), Gumbel>Joe (lower): True


**To the Student-$t$ degrees of freedom.** The Student-$t$ row depends on $\nu$, which the thesis fixes at 4 for the illustration only. Its tail dependence is strong for any finite $\nu$, but the size changes substantially.

In [10]:
rows = []
for nu in (2, 4, 8, 30):
    b = build(F.student, 0, TAU, nu)
    rows.append({"nu": nu, "lambda": taildep(b)[0],
                 "lower P(U2<1-q|U1<1-q)": chi_lower(b, Q)[0], "upper P(U2>q|U1>q)": chi_upper(b, Q)[0]})
pd.DataFrame(rows).set_index("nu")

,lambda,lower P(U2<1-q|U1<1-q),upper P(U2>q|U1>q)
nu,,,
2,0.525,0.545,0.545
4,0.397,0.482,0.482
8,0.245,0.442,0.442
30,0.028,0.411,0.411


## Notes

* The family parameters here are set from Kendall's $\tau$ only to compare shapes; in the thesis the parameters are estimated by Maximum Pseudo-Likelihood.
* Tail orders for Clayton (upper), Joe (lower) and Frank are $2$, and for Gumbel (lower) $2^{1/\theta}$; these follow from the closed forms of the copulas and are confirmed by the numerical slopes above.
* If a different tau, threshold or number of degrees of freedom is wanted, change the constants at the top and run all cells; the check in Section 6 compares against the numbers printed in the text at the default settings, so it only applies to `TAU = 0.5`, `Q = 0.95`.